This notebook introduces `allocate`, the SysML v2 relationship that assigns a behavioral element to a structural part; after running it you can express which hardware component is responsible for which function.

The cumulative model has an `ApplyHeat` action definition (Ch4) and a `HeatingSystem` part definition (Ch1). They are related by design intent but not yet formally connected. `allocate` makes that connection explicit: it states that the heating subsystem is the structural locus of the heating action.

`allocate X to Y` creates an `AllocationUsage` element. OpenSysML stores it in the model graph; `model.to_api_json()` exposes it alongside `FlowUsage` elements with connector endpoints.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")

In [ ]:
# Negative control: allocating an undefined symbol raises "unresolved reference".
# Both the source and target of allocate must be defined in scope.
bad_source = """
package BadAlloc {
    allocate UndefinedAction to HeatingSystem;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
import json, warnings

# AllocationUsage elements are in the JSON export (not in model.query())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    data = json.loads(model.to_api_json().content)

by_id = {e["@id"]: e for e in data if "@id" in e}
for elem in data:
    if elem.get("@type") == "AllocationUsage":
        ends = elem.get("connectorEnd", [])
        if len(ends) == 2:
            src = by_id.get(ends[0]["@id"], {}).get("sysx:sourceText", "?")
            tgt = by_id.get(ends[1]["@id"], {}).get("sysx:sourceText", "?")
            print(f"allocate {src!r} to {tgt!r}")

The `allocate` relationship in SysML v2 (A-F) is parsed and stored in the OpenSysML element graph (O-S); querying the JSON export and reading `sysx:sourceText` from each connector endpoint reveals the assignment as `'ApplyHeat'` → `'HeatingSystem'` (E).

Try the chapter exercise in `exercises/ch05/exercise.ipynb`: add an `allocate` statement assigning your `Brew` action to a `BrewUnit` part, then confirm the allocation appears in the JSON export.